# Notebook 03: Dijkstra's Algorithm and A* Search

## Overview

This notebook explores weighted pathfinding algorithms that handle grids with different terrain costs:
- **Dijkstra's Algorithm**: Finds optimal paths in weighted graphs
- **A* Search**: Enhanced Dijkstra with heuristic guidance for better performance

## Learning Objectives

By the end of this notebook, you will:
1. Understand why weighted graphs require different algorithms than BFS
2. Learn how Dijkstra guarantees optimal paths in weighted graphs
3. Understand how A* uses heuristics to guide search
4. Compare efficiency of Dijkstra vs A* with different heuristics
5. Recognize the importance of admissible heuristics

## Key Concepts

### Dijkstra's Algorithm
- Expands nodes in order of increasing cost from start
- Uses a **priority queue** ordered by accumulated cost g(n)
- **Guarantees optimal path** in weighted graphs
- Can be thought of as "uniform-cost search"
- Time Complexity: O((V + E) log V) with binary heap

### A* Search
- Combines Dijkstra with a heuristic function
- Expands nodes based on f(n) = g(n) + h(n) where:
  - g(n) = actual cost from start to n
  - h(n) = estimated cost from n to goal (heuristic)
- **Guarantees optimal path** if heuristic is admissible
- Generally more efficient than Dijkstra
- Time Complexity: O((V + E) log V), but explores fewer nodes in practice

### Admissible Heuristics
A heuristic h(n) is **admissible** if it never overestimates the true cost to reach the goal.
- h(n) ≤ actual cost from n to goal
- Examples: Manhattan distance, Euclidean distance
- Admissible heuristics guarantee A* finds optimal paths

## Setup and Imports

In [ ]:
# Add parent directory to path for imports
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# Core imports
from src.pathfinding_lab.core.grid import Grid
from src.pathfinding_lab.core.types import MovementMode
from src.pathfinding_lab.algorithms.dijkstra import dijkstra
from src.pathfinding_lab.algorithms.astar import astar
from src.pathfinding_lab.algorithms.bfs import bfs

# Heuristic functions
from src.pathfinding_lab.heuristics.manhattan import manhattan_distance
from src.pathfinding_lab.heuristics.euclidean import euclidean_distance
from src.pathfinding_lab.heuristics.octile import octile_distance
from src.pathfinding_lab.heuristics.chebyshev import chebyshev_distance

# Visualization
from src.pathfinding_lab.visualization.grid_plot import create_grid_plot
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-darkgrid')
print("✓ Imports successful!")

## Part 1: Why BFS Fails on Weighted Grids

Let's first understand why BFS doesn't work for grids with diagonal movement (which has different costs for cardinal vs diagonal moves).

In [ ]:
# Create a simple 10x10 grid with 8-directional movement
grid = Grid(width=10, height=10, obstacle_density=0.1,
            movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)

start = (1, 1)
goal = (8, 8)
grid.generate_obstacles(start, goal)

print("Grid with 8-directional movement:")
print(f"- Cardinal move cost: 1.0")
print(f"- Diagonal move cost: {np.sqrt(2):.3f}")
print(f"\nThis is a WEIGHTED graph because edges have different costs!")

In [ ]:
# Run BFS, Dijkstra, and A* on the same grid
result_bfs = bfs(grid, start, goal)
result_dijkstra = dijkstra(grid, start, goal)
result_astar = astar(grid, start, goal, manhattan_distance)

# Compare path costs
print("\n" + "="*70)
print("PATH COST COMPARISON (Lower is better)")
print("="*70)
print(f"{'Algorithm':<20} {'Path Length':<15} {'Path Cost':<20} {'Optimal?'}")
print("-"*70)
print(f"{'BFS':<20} {result_bfs.path_length:<15} {result_bfs.path_cost:<20.3f} {'❌ No'}")
print(f"{'Dijkstra':<20} {result_dijkstra.path_length:<15} {result_dijkstra.path_cost:<20.3f} {'✓ Yes'}")
print(f"{'A* (Manhattan)':<20} {result_astar.path_length:<15} {result_astar.path_cost:<20.3f} {'✓ Yes'}")
print("="*70)
print("\nNote: BFS minimizes PATH LENGTH, not PATH COST!")
print("Dijkstra and A* both find the path with minimum COST.")

## Part 2: Dijkstra's Algorithm in Action

In [ ]:
# Create a larger grid for better visualization
large_grid = Grid(width=20, height=20, obstacle_density=0.15,
                  movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=123)

large_start = (2, 2)
large_goal = (17, 17)
large_grid.generate_obstacles(large_start, large_goal)

# Run Dijkstra
dijkstra_result = dijkstra(large_grid, large_start, large_goal)

print(f"Algorithm: {dijkstra_result.algorithm_name}")
print(f"Success: {dijkstra_result.success}")
print(f"Path length: {dijkstra_result.path_length} steps")
print(f"Path cost: {dijkstra_result.path_cost:.3f}")
print(f"Nodes visited: {dijkstra_result.nodes_visited}")
print(f"Runtime: {dijkstra_result.runtime_ms:.2f} ms")

In [ ]:
# Visualize Dijkstra result
fig = create_grid_plot(large_grid, large_start, large_goal, dijkstra_result, figsize=(14, 12))
plt.show()

print("\nNotice how Dijkstra explores nodes uniformly in all directions,")
print("like ripples spreading out from the start point.")

## Part 3: A* Search with Different Heuristics

A* uses a heuristic to guide the search toward the goal. Let's compare different heuristics.

In [ ]:
# Test A* with different heuristics
heuristics = [
    ("Manhattan", manhattan_distance),
    ("Euclidean", euclidean_distance),
    ("Octile", octile_distance),
    ("Chebyshev", chebyshev_distance),
]

results = {}

print("Running A* with different heuristics...\n")

for name, heuristic in heuristics:
    result = astar(large_grid, large_start, large_goal, heuristic)
    results[name] = result
    print(f"{name:12s}: Cost={result.path_cost:.3f}, Visited={result.nodes_visited:4d}, Time={result.runtime_ms:.2f}ms")

print(f"\nDijkstra   : Cost={dijkstra_result.path_cost:.3f}, Visited={dijkstra_result.nodes_visited:4d}, Time={dijkstra_result.runtime_ms:.2f}ms")

In [ ]:
# Visualize A* with different heuristics
fig, axes = plt.subplots(2, 2, figsize=(20, 18))
axes = axes.flatten()

from matplotlib.colors import ListedColormap

def plot_on_axis(ax, grid, start, goal, result, title):
    vis_grid = np.zeros((grid.height, grid.width))
    
    for obstacle in grid.obstacles:
        vis_grid[obstacle[0], obstacle[1]] = 1
    
    if result and result.visited_order:
        for pos in result.visited_order:
            if pos != start and pos != goal:
                vis_grid[pos[0], pos[1]] = 2
    
    if result and result.path:
        for pos in result.path:
            if pos != start and pos != goal:
                vis_grid[pos[0], pos[1]] = 3
    
    vis_grid[start[0], start[1]] = 4
    vis_grid[goal[0], goal[1]] = 5
    
    colors = ['white', 'black', 'lightblue', 'yellow', 'green', 'red']
    cmap = ListedColormap(colors)
    ax.imshow(vis_grid, cmap=cmap, vmin=0, vmax=5)
    
    ax.set_xticks(np.arange(-0.5, grid.width, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, grid.height, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    ax.tick_params(which='minor', size=0)
    
    stats = f"Visited: {result.nodes_visited}, Cost: {result.path_cost:.3f}, Time: {result.runtime_ms:.2f}ms"
    ax.set_title(f"{title}\n{stats}", fontsize=11, fontweight='bold')

for idx, (name, result) in enumerate(results.items()):
    plot_on_axis(axes[idx], large_grid, large_start, large_goal, result, f"A* with {name} Heuristic")

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- All heuristics find the same optimal path (same cost)")
print("- Different heuristics visit different numbers of nodes")
print("- Better heuristics = more focused search = fewer nodes visited")

## Part 4: Comparing Dijkstra vs A*

In [ ]:
# Side-by-side comparison: Dijkstra vs A* (Euclidean)
fig, axes = plt.subplots(1, 2, figsize=(22, 10))

plot_on_axis(axes[0], large_grid, large_start, large_goal, dijkstra_result, 
             "Dijkstra's Algorithm")
plot_on_axis(axes[1], large_grid, large_start, large_goal, results["Euclidean"], 
             "A* with Euclidean Heuristic")

plt.tight_layout()
plt.show()

print("\nKey Difference:")
print(f"Dijkstra visited:  {dijkstra_result.nodes_visited} nodes (explores uniformly)")
print(f"A* visited:        {results['Euclidean'].nodes_visited} nodes (guided by heuristic)")
reduction = (1 - results['Euclidean'].nodes_visited / dijkstra_result.nodes_visited) * 100
print(f"\nA* reduces node exploration by {reduction:.1f}%!")

## Part 5: Performance Benchmarking

In [ ]:
# Benchmark across different grid sizes
grid_sizes = [10, 15, 20, 25, 30, 35]

dijkstra_times = []
dijkstra_visited = []
astar_times = []
astar_visited = []

print("Benchmarking across different grid sizes...\n")

for size in grid_sizes:
    test_grid = Grid(width=size, height=size, obstacle_density=0.2,
                     movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)
    test_start = (1, 1)
    test_goal = (size-2, size-2)
    test_grid.generate_obstacles(test_start, test_goal)
    
    # Run Dijkstra
    d_result = dijkstra(test_grid, test_start, test_goal)
    dijkstra_times.append(d_result.runtime_ms)
    dijkstra_visited.append(d_result.nodes_visited)
    
    # Run A* with Euclidean
    a_result = astar(test_grid, test_start, test_goal, euclidean_distance)
    astar_times.append(a_result.runtime_ms)
    astar_visited.append(a_result.nodes_visited)
    
    print(f"Size {size:2d}x{size:2d}: Dijkstra={d_result.runtime_ms:6.2f}ms, A*={a_result.runtime_ms:6.2f}ms")

print("\nBenchmarking complete!")

In [ ]:
# Plot performance comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Runtime comparison
axes[0].plot(grid_sizes, dijkstra_times, marker='o', linewidth=2.5, markersize=8, label='Dijkstra')
axes[0].plot(grid_sizes, astar_times, marker='s', linewidth=2.5, markersize=8, label='A* (Euclidean)')
axes[0].set_xlabel('Grid Size (N x N)', fontsize=12)
axes[0].set_ylabel('Runtime (ms)', fontsize=12)
axes[0].set_title('Runtime Comparison', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Nodes visited comparison
axes[1].plot(grid_sizes, dijkstra_visited, marker='o', linewidth=2.5, markersize=8, label='Dijkstra')
axes[1].plot(grid_sizes, astar_visited, marker='s', linewidth=2.5, markersize=8, label='A* (Euclidean)')
axes[1].set_xlabel('Grid Size (N x N)', fontsize=12)
axes[1].set_ylabel('Nodes Visited', fontsize=12)
axes[1].set_title('Nodes Visited Comparison', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Efficiency ratio (A* / Dijkstra)
efficiency_ratio = [a / d for a, d in zip(astar_visited, dijkstra_visited)]
axes[2].plot(grid_sizes, efficiency_ratio, marker='D', linewidth=2.5, markersize=8, color='green')
axes[2].axhline(y=1.0, color='red', linestyle='--', label='No improvement')
axes[2].set_xlabel('Grid Size (N x N)', fontsize=12)
axes[2].set_ylabel('Ratio (A* / Dijkstra)', fontsize=12)
axes[2].set_title('A* Efficiency Ratio\n(Lower is Better)', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].set_ylim([0, 1.1])

plt.tight_layout()
plt.show()

avg_ratio = np.mean(efficiency_ratio)
print(f"\nOn average, A* visits only {avg_ratio*100:.1f}% of the nodes that Dijkstra visits!")

## Part 6: Heuristic Quality Comparison

Let's analyze which heuristic works best for 8-directional movement.

In [ ]:
# Compare all heuristics on multiple test cases
test_cases = [
    ("Small", 15, 0.1),
    ("Medium", 25, 0.15),
    ("Large", 35, 0.2),
]

heuristic_stats = {name: {'visited': [], 'time': []} for name, _ in heuristics}

for case_name, size, density in test_cases:
    test_grid = Grid(width=size, height=size, obstacle_density=density,
                     movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=42)
    test_start = (1, 1)
    test_goal = (size-2, size-2)
    test_grid.generate_obstacles(test_start, test_goal)
    
    print(f"\n{case_name} Grid ({size}x{size}):")
    
    for h_name, h_func in heuristics:
        result = astar(test_grid, test_start, test_goal, h_func)
        heuristic_stats[h_name]['visited'].append(result.nodes_visited)
        heuristic_stats[h_name]['time'].append(result.runtime_ms)
        print(f"  {h_name:12s}: Visited={result.nodes_visited:4d}, Time={result.runtime_ms:6.2f}ms")

In [ ]:
# Plot heuristic comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

case_names = [name for name, _, _ in test_cases]
x_pos = np.arange(len(case_names))
width = 0.2

# Nodes visited comparison
for idx, (h_name, _) in enumerate(heuristics):
    visited = heuristic_stats[h_name]['visited']
    axes[0].bar(x_pos + idx*width, visited, width, label=h_name)

axes[0].set_xlabel('Test Case', fontsize=12)
axes[0].set_ylabel('Nodes Visited', fontsize=12)
axes[0].set_title('Nodes Visited by Heuristic', fontsize=14, fontweight='bold')
axes[0].set_xticks(x_pos + width * 1.5)
axes[0].set_xticklabels(case_names)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Runtime comparison
for idx, (h_name, _) in enumerate(heuristics):
    times = heuristic_stats[h_name]['time']
    axes[1].bar(x_pos + idx*width, times, width, label=h_name)

axes[1].set_xlabel('Test Case', fontsize=12)
axes[1].set_ylabel('Runtime (ms)', fontsize=12)
axes[1].set_title('Runtime by Heuristic', fontsize=14, fontweight='bold')
axes[1].set_xticks(x_pos + width * 1.5)
axes[1].set_xticklabels(case_names)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nWhich heuristic performs best for 8-directional movement?")
print("Hint: Octile distance is designed specifically for 8-directional grids!")

## Exercise 1: Test with 4-Directional Movement

How do the heuristics compare when only cardinal (4-directional) movement is allowed?

In [ ]:
# Your code here
# Create a grid with FOUR_DIRECTIONAL movement
# Test all heuristics and compare results

grid_4dir = Grid(width=25, height=25, obstacle_density=0.15,
                 movement_mode=MovementMode.FOUR_DIRECTIONAL, random_seed=42)

# Generate obstacles and run A* with different heuristics
# Which heuristic works best for 4-directional movement?
# ...

## Exercise 2: Inadmissible Heuristic

What happens if we use an inadmissible heuristic (one that overestimates)?

In [ ]:
# Create an inadmissible heuristic by multiplying Manhattan distance by 2
def inadmissible_heuristic(pos1, pos2):
    return manhattan_distance(pos1, pos2) * 2.0

# Test on a grid
test_grid = Grid(width=20, height=20, obstacle_density=0.15,
                 movement_mode=MovementMode.EIGHT_DIRECTIONAL, random_seed=100)
test_start = (2, 2)
test_goal = (17, 17)
test_grid.generate_obstacles(test_start, test_goal)

# Compare admissible vs inadmissible
result_admissible = astar(test_grid, test_start, test_goal, manhattan_distance)
result_inadmissible = astar(test_grid, test_start, test_goal, inadmissible_heuristic)
result_optimal = dijkstra(test_grid, test_start, test_goal)

print("Optimal cost (Dijkstra):    ", result_optimal.path_cost)
print("A* with admissible:         ", result_admissible.path_cost)
print("A* with inadmissible:       ", result_inadmissible.path_cost)
print("\nDoes the inadmissible heuristic still find the optimal path?")

## Key Takeaways

### Dijkstra's Algorithm:
- **Always finds the optimal path** in weighted graphs
- Explores nodes uniformly based on cost from start
- No domain knowledge required (no heuristic needed)
- Can be slower than A* because it explores more nodes

### A* Search:
- **Finds optimal path if heuristic is admissible**
- Uses heuristic to guide search toward goal
- Typically much faster than Dijkstra (visits fewer nodes)
- Heuristic quality directly impacts performance

### Choosing Heuristics:
- **4-Directional**: Manhattan distance is best (exact)
- **8-Directional**: Octile distance is best (tighter estimate)
- **Any movement**: Euclidean distance is safe and good
- **Important**: Always ensure admissibility for optimality!

### When to Use Each:
- **Dijkstra**: When you have no domain knowledge or need to find distances to all nodes
- **A***: When you have a good admissible heuristic and a single goal
- **Both**: Guarantee optimal paths unlike BFS/DFS on weighted graphs

## Next Steps

In the next notebook (04_heuristics.ipynb), we'll:
- Deep dive into all 5 heuristic functions
- Prove admissibility mathematically
- Visualize heuristic values on grids
- Learn when each heuristic is best

## Further Reading

- [Wikipedia: Dijkstra's Algorithm](https://en.wikipedia.org/wiki/Dijkstra%27s_algorithm)
- [Wikipedia: A* Search](https://en.wikipedia.org/wiki/A*_search_algorithm)
- [Amit's A* Pages](http://theory.stanford.edu/~amitp/GameProgramming/) (Excellent resource!)
- Introduction to Algorithms (CLRS), Chapter 24